# Step 4 (tiếp) – Option C: Traditional Relational Database (SQLite)



In [1]:
import sqlite3
import pandas as pd

DB_PATH = 'pharmacity.db'
CSV_PATH = 'final_master_dataset.csv'

df = pd.read_csv(CSV_PATH)
df['PurchaseDate'] = pd.to_datetime(df['PurchaseDate']).dt.strftime('%Y-%m-%d')  # SQLite lưu ngày dạng TEXT ISO
print('Shape:', df.shape)
df.head(3)

Shape: (1000, 20)


,OrderID,CustomerID,CustomerName,Age,Gender,ProductID,UnitPrice,Quantity,TotalAmount,PurchaseDate,City,PaymentMethod,ProductName,Brand,Category,Price,Rating,Source,Currency,Price_VND_ref
0,ORD00001,C0001,An Vũ,35,Nữ,P18999,194400,1,194400,2026-07-09,TP. Hồ Chí Minh,Tiền mặt,Dầu Gội TRESEMME Salon Rebond Giúp Giảm Gãy Rụ...,Tresemme,Dầu gội đầu,194400.0,4.70,Pharmacity_VN,VND,194400.0
1,ORD00002,C0002,Chị Lan Trần,29,Nam,P13420,61000,1,61000,2026-05-16,Hà Nội,Ví điện tử,Sáp dưỡng ẩm Vaseline Pure Petroleum Jelly Ori...,Vaseline,Dưỡng thể & massage,61000.0,4.77,Pharmacity_VN,VND,61000.0
2,ORD00003,C0003,Khoa Bùi,36,Nam,P12848,189660,2,379320,2026-02-12,Hà Nội,Thẻ ngân hàng,Tã quần mỏng thoáng size L cho người lớn Caryn...,Caryn,Tã dán,189660.0,4.86,Pharmacity_VN,VND,189660.0


## 1. Kết nối & tạo bảng 


In [2]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute('DROP TABLE IF EXISTS master_dataset')
cur.execute('''
CREATE TABLE master_dataset (
    OrderID        TEXT PRIMARY KEY,
    CustomerID     TEXT NOT NULL,
    CustomerName   TEXT,
    Age            INTEGER CHECK (Age > 0),
    Gender         TEXT,
    ProductID      TEXT NOT NULL,
    UnitPrice      INTEGER,
    Quantity       INTEGER CHECK (Quantity > 0),
    TotalAmount    INTEGER,
    PurchaseDate   TEXT,
    City           TEXT,
    PaymentMethod  TEXT,
    ProductName    TEXT,
    Brand          TEXT,     
    Category       TEXT,
    Price          REAL,
    Rating         REAL,
    Source         TEXT,
    Currency       TEXT,
    Price_VND_ref  REAL
)
''')
conn.commit()
print('Đã tạo bảng master_dataset')

Đã tạo bảng master_dataset


## 2. Nạp dữ liệu
`NaN` của pandas được đổi thành `None` để lưu thành `NULL` trong SQL.

In [3]:
records = df.astype(object).where(df.notna(), None).values.tolist()
placeholders = ','.join(['?'] * len(df.columns))

cur.executemany(f'INSERT INTO master_dataset ({",".join(df.columns)}) VALUES ({placeholders})', records)
conn.commit()

print('Inserted rows:', cur.execute('SELECT COUNT(*) FROM master_dataset').fetchone()[0])

Inserted rows: 1000


## 3. Tạo index

In [4]:
cur.execute('CREATE INDEX IF NOT EXISTS idx_customer ON master_dataset(CustomerID)')
cur.execute('CREATE INDEX IF NOT EXISTS idx_category ON master_dataset(Category)')
cur.execute('CREATE INDEX IF NOT EXISTS idx_city_date ON master_dataset(City, PurchaseDate)')
conn.commit()
pd.read_sql("SELECT name, tbl_name FROM sqlite_master WHERE type='index'", conn)

,name,tbl_name
0,sqlite_autoindex_master_dataset_1,master_dataset
1,idx_customer,master_dataset
2,idx_category,master_dataset
3,idx_city_date,master_dataset


## 4. Truy vấn kiểm tra bằng SQL

In [5]:
q = '''
SELECT Category,
       COUNT(*)              AS so_don,
       SUM(TotalAmount)      AS doanh_thu,
       ROUND(AVG(Rating), 2) AS rating_tb
FROM master_dataset
GROUP BY Category
ORDER BY doanh_thu DESC
LIMIT 10
'''
pd.read_sql(q, conn)

,Category,so_don,doanh_thu,rating_tb
0,Dầu gội đầu,97,27758770,4.78
1,Băng vệ sinh,186,16962340,4.79
2,Dung dịch vệ sinh phụ nữ,57,14978200,4.73
3,Sữa tắm,62,13685900,4.77
4,Nước súc miệng,70,13349750,4.77
5,Kem đánh răng,87,12455190,4.76
6,Khử mùi cho nam,58,10860160,4.76
7,Bàn chải đánh răng,83,8301850,4.76
8,Dầu xả,25,7221000,4.81
9,Dưỡng thể & massage,33,6295800,4.74


In [6]:
q = '''
SELECT City, PaymentMethod, COUNT(*) AS so_don
FROM master_dataset
GROUP BY City, PaymentMethod
ORDER BY City, so_don DESC
'''
pd.read_sql(q, conn).head(10)

,City,PaymentMethod,so_don
0,Cần Thơ,COD,30
1,Cần Thơ,Ví điện tử,27
2,Cần Thơ,Tiền mặt,21
3,Cần Thơ,Thẻ ngân hàng,21
4,Hà Nội,Thẻ ngân hàng,88
5,Hà Nội,Ví điện tử,82
6,Hà Nội,COD,73
7,Hà Nội,Tiền mặt,64
8,Hải Phòng,COD,34
9,Hải Phòng,Thẻ ngân hàng,19


## 5. Đọc ngược dữ liệu từ SQLite về Python (chuẩn bị cho EDA)
Đề yêu cầu: kết nối CSDL bằng code → load lại vào Python → làm EDA.

In [7]:
df_sql = pd.read_sql('SELECT * FROM master_dataset', conn, parse_dates=['PurchaseDate'])
print('Shape:', df_sql.shape)
print('\nMissing values:')
print(df_sql.isna().sum()[lambda s: s > 0])
df_sql.head()

Shape: (1000, 20)

Missing values:
Brand    4
dtype: int64


,OrderID,CustomerID,CustomerName,Age,Gender,ProductID,UnitPrice,Quantity,TotalAmount,PurchaseDate,City,PaymentMethod,ProductName,Brand,Category,Price,Rating,Source,Currency,Price_VND_ref
0,ORD00001,C0001,An Vũ,35,Nữ,P18999,194400,1,194400,2026-07-09,TP. Hồ Chí Minh,Tiền mặt,Dầu Gội TRESEMME Salon Rebond Giúp Giảm Gãy Rụ...,Tresemme,Dầu gội đầu,194400.0,4.70,Pharmacity_VN,VND,194400.0
1,ORD00002,C0002,Chị Lan Trần,29,Nam,P13420,61000,1,61000,2026-05-16,Hà Nội,Ví điện tử,Sáp dưỡng ẩm Vaseline Pure Petroleum Jelly Ori...,Vaseline,Dưỡng thể & massage,61000.0,4.77,Pharmacity_VN,VND,61000.0
2,ORD00003,C0003,Khoa Bùi,36,Nam,P12848,189660,2,379320,2026-02-12,Hà Nội,Thẻ ngân hàng,Tã quần mỏng thoáng size L cho người lớn Caryn...,Caryn,Tã dán,189660.0,4.86,Pharmacity_VN,VND,189660.0
3,ORD00004,C0004,Ngọc Trần,45,Nam,P30966,40000,2,80000,2026-03-31,Hải Phòng,Thẻ ngân hàng,Kotex BVS Mặt Bông Cotton SMC 24cm 8M,Kotex,Băng vệ sinh,40000.0,4.53,Pharmacity_VN,VND,40000.0
4,ORD00005,C0005,Chị Vân Hoàng,28,Nữ,P21174,203000,1,203000,2026-09-02,TP. Hồ Chí Minh,Tiền mặt,Serum Dưỡng Tóc ELLIPS Giúp Tóc Phục Hồi Hư Tổ...,Ellips,Dưỡng tóc,203000.0,4.89,Pharmacity_VN,VND,203000.0


In [8]:
conn.close()
print('Đã đóng kết nối. File CSDL:', DB_PATH)

Đã đóng kết nối. File CSDL: pharmacity.db
